# Empirical Ablation Study: Data Preprocessing Filters

Comparing 7 pipelines (P1–P7) using different gatekeeping strategies before a **MultinomialNB** classifier.

In [1]:
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    dataset_path = "/content/drive/MyDrive/datasets"
    print("environtment: colab")
except ImportError:
    dataset_path = "./datasets"
    print("environtment: local")

environtment: local


In [2]:
import pandas as pd
import glob
import numpy as np
import ast
import csv
import os
import re
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
all_files = glob.glob(os.path.join(dataset_path, "*.csv"))

df_raw = pd.concat([
    pd.read_csv(
        f,
        engine="python",
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip"
    )
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df_raw)}")
print(f"Dari {len(all_files)} file: {[os.path.basename(f) for f in all_files]}")
df_raw.info()

Total rows: 7736
Dari 1 file: ['FinalFile_new(banget).csv']
<class 'pandas.DataFrame'>
RangeIndex: 7736 entries, 0 to 7735
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                7736 non-null   float64
 1   title             7736 non-null   str    
 2   search_role       7736 non-null   str    
 3   job_level         7736 non-null   str    
 4   company           7736 non-null   str    
 5   location          7736 non-null   str    
 6   salary_avg        1649 non-null   float64
 7   extracted_skills  7736 non-null   str    
 8   skills_count      7736 non-null   int64  
 9   job_description   7730 non-null   str    
 10  job_url           7736 non-null   str    
 11  scraped_at        5728 non-null   str    
 12  source            7736 non-null   str    
 13  salary_min        0 non-null      float64
 14  salary_max        0 non-null      float64
 15  salary_display    7736 non-null   str   

## Base Preprocessing
Drop irrelevant columns, remove duplicates, fill nulls — identical to `main.ipynb`.

In [4]:
DROP_COLS = [
    'salary', 'search_role_raw', 'job_level', 'location_raw',
    'salary_display', 'salary_min', 'salary_max', 'salary_avg',
    'company', 'location', 'job_url', 'search_location', 'scraped_at',
]

df_base = df_raw.copy()
df_base = df_base.drop(columns=DROP_COLS, errors='ignore')
df_base = df_base.drop_duplicates()
df_base.dropna(subset=['extracted_skills', 'search_role'], inplace=True)
df_base['job_description'] = df_base['job_description'].fillna('')
df_base['title'] = df_base['title'].fillna('')
df_base = df_base.reset_index(drop=True)

df_base['search_role'] = df_base['search_role'].replace({'Full stack Developer': 'Fullstack Developer'})
ROLES_TO_DROP = [
    'Software Engineer', 'Web Developer', 'IT Support', 'IT',
    'Developer', 'Software', 'System Analyst', 'Progammer', 'Software Developer',
]
df_base = df_base[~df_base['search_role'].isin(ROLES_TO_DROP)].reset_index(drop=True)

print(f"Rows after base preprocessing: {len(df_base)}")
print(df_base['search_role'].value_counts())

Rows after base preprocessing: 7300
search_role
Fullstack Developer    1806
DevOps Engineer        1251
Data Analyst            950
Data Engineer           886
Backend Developer       795
Frontend Developer      572
ML Engineer             548
Data Scientist          492
Name: count, dtype: int64


## Feature Engineering Helpers

In [5]:
def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        if isinstance(result, list) and result:
            return ' '.join(
                s.lower().replace(' ', '_').replace('/', '_')
                for s in result
            )
        return ''
    except Exception:
        return ''

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    return ' '.join(text.split()).lower()

def clean_jd(text: str, max_chars: int = 1000) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text[:max_chars]
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    tokens = [t for t in text.lower().split() if len(t) >= 4]
    return ' '.join(tokens)

df_base['skills_text']   = df_base['extracted_skills'].apply(parse_skills)
df_base['title_text']    = df_base['title'].apply(clean_text)
df_base['jd_text']       = df_base['job_description'].apply(clean_jd)
df_base['combined_text'] = (
    df_base['skills_text'] + ' [SEP] ' +
    df_base['title_text']  + ' [SEP] ' +
    df_base['jd_text']
)

df_base = df_base[df_base['skills_text'].str.strip() != ''].reset_index(drop=True)
df_base = df_base[df_base['combined_text'].str.strip() != ''].reset_index(drop=True)

print(f"Rows after feature engineering: {len(df_base)}")
print('\nSample combined_text:')
print(df_base['combined_text'].iloc[0][:300])

Rows after feature engineering: 4449

Sample combined_text:
angular aws azure ci_cd django express fastapi fiber gcp git java javascript kafka kotlin mongodb mysql postgresql python react redis solid spring_boot typescript vue [SEP] fullstack engineer sde 1 [SEP] about about design implement small medium sized features across frontend backend systems build h


## Reference Standard & Cosine Threshold

In [6]:
REFERENCE_STANDARD = {
    'Frontend Developer'  : 'react vue angular typescript javascript css html ui frontend web',
    'Backend Developer'   : 'java golang python php nodejs api backend rest microservices database',
    'Fullstack Developer' : 'fullstack full stack web software react nodejs postgresql docker',
    'Data Analyst'        : 'data analyst analytics sql tableau bi business intelligence reporting',
    'Data Scientist'      : 'data scientist machine learning ai python tensorflow scikit keras',
    'Data Engineer'       : 'data engineer pipeline spark airflow kafka bigquery cloud etl',
    'DevOps Engineer'     : 'devops sre infrastructure kubernetes docker aws azure linux terraform',
    'ML Engineer'         : 'ml machine learning nlp computer vision pytorch tensorflow llm ai',
}

COSINE_THRESHOLD = 0.05
print("Reference Standard roles :", list(REFERENCE_STANDARD.keys()))
print(f"Cosine similarity threshold: {COSINE_THRESHOLD}")

Reference Standard roles : ['Frontend Developer', 'Backend Developer', 'Fullstack Developer', 'Data Analyst', 'Data Scientist', 'Data Engineer', 'DevOps Engineer', 'ML Engineer']
Cosine similarity threshold: 0.05


## Shared Classifier Helper

Key robustness fixes applied here:
- **Drop sparse classes**: any role with fewer than `min_class_samples` rows after filtering is removed before splitting, preventing `stratify` from crashing.
- **Adaptive split**: if remaining classes still can't support stratified splitting, fall back to a plain (non-stratified) split.

In [7]:
MIN_CLASS_SAMPLES = 2   # minimum rows a class must have to be kept after filtering

def safe_train_evaluate(df_filtered: pd.DataFrame,
                        pipeline_name: str,
                        test_size: float = 0.20,
                        random_state: int = 42) -> dict:
    """
    Train MultinomialNB on df_filtered and return accuracy.

    Robustness measures:
    1. Drop any role class that has fewer than MIN_CLASS_SAMPLES rows
       (a cosine filter can cut a role down to 1 sample, breaking stratify).
    2. Try stratified train/test split; fall back to non-stratified if still
       impossible (e.g. a class has only 1 sample left after dropping).
    """
    if len(df_filtered) < 10:
        print(f"[{pipeline_name}] Skipped — too few rows ({len(df_filtered)}).")
        return {'pipeline': pipeline_name, 'remaining_rows': len(df_filtered),
                'rows_used_for_training': 0, 'accuracy': None}

    # ── 1. Remove classes with too few samples ──────────────────────────────
    role_counts = df_filtered['search_role'].value_counts()
    valid_roles = role_counts[role_counts >= MIN_CLASS_SAMPLES].index
    dropped_roles = set(df_filtered['search_role'].unique()) - set(valid_roles)
    if dropped_roles:
        print(f"  [{pipeline_name}] Dropping sparse roles (< {MIN_CLASS_SAMPLES} samples): {dropped_roles}")
    df_clean = df_filtered[df_filtered['search_role'].isin(valid_roles)].reset_index(drop=True)

    if len(df_clean) < 10:
        print(f"[{pipeline_name}] Skipped — too few rows after sparse-class removal ({len(df_clean)}).")
        return {'pipeline': pipeline_name, 'remaining_rows': len(df_filtered),
                'rows_used_for_training': len(df_clean), 'accuracy': None}

    le = LabelEncoder()
    y  = le.fit_transform(df_clean['search_role'])
    X  = df_clean['combined_text'].tolist()

    # ── 2. Stratified split with non-stratified fallback ────────────────────
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )
    except ValueError as e:
        print(f"  [{pipeline_name}] Stratified split failed ({e}). Falling back to plain split.")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )

    # ── 3. TF-IDF + MultinomialNB ────────────────────────────────────────────
    vec = TfidfVectorizer(max_features=1500, ngram_range=(1, 2),
                          min_df=2, sublinear_tf=True)
    X_train_v = vec.fit_transform(X_train)
    X_test_v  = vec.transform(X_test)

    clf = MultinomialNB()
    clf.fit(X_train_v, y_train)

    accuracy = accuracy_score(y_test, clf.predict(X_test_v))
    print(f"[{pipeline_name}]  Rows after filter: {len(df_filtered):,}  |  Used for training: {len(df_clean):,}  |  Accuracy: {accuracy:.4f}")

    return {
        'pipeline'              : pipeline_name,
        'remaining_rows'        : len(df_filtered),
        'rows_used_for_training': len(df_clean),
        'accuracy'              : round(accuracy, 4),
    }

## Filter Implementations

In [8]:
def cosine_filter(df: pd.DataFrame,
                   ngram_range: tuple,
                   threshold: float = COSINE_THRESHOLD) -> pd.DataFrame:
    """
    Keep a row only if cosine_similarity(combined_text, role_reference) >= threshold.
    Uses per-row TF-IDF fit (combined_text + reference text as a 2-doc corpus).
    """
    keep_mask = []
    for _, row in df.iterrows():
        ref_text = REFERENCE_STANDARD.get(row['search_role'], row['search_role'])
        corpus   = [row['combined_text'], ref_text]
        vec      = TfidfVectorizer(ngram_range=ngram_range)
        try:
            mat   = vec.fit_transform(corpus)
            score = cosine_similarity(mat[0:1], mat[1:2])[0][0]
        except Exception:
            score = 0.0
        keep_mask.append(score >= threshold)
    return df[keep_mask].reset_index(drop=True)


KEYWORD_MAP_P7 = {
    'frontend'        : ['frontend', 'front-end', 'front end', 'react', 'vue', 'angular', 'ui', 'web'],
    'backend'         : ['backend', 'back-end', 'back end', 'java', 'golang', 'python', 'php', 'node', 'api'],
    'fullstack'       : ['fullstack', 'full-stack', 'full stack', 'software', 'web', 'programmer'],
    'data analyst'    : ['analyst', 'analytics', 'data', 'bi ', 'business intelligence'],
    'data scientist'  : ['scientist', 'data', 'machine learning', 'ai '],
    'data engineer'   : ['engineer', 'data', 'pipeline', 'cloud', 'big data'],
    'devops engineer' : ['devops', 'sre', 'reliability', 'infrastructure', 'cloud', 'aws', 'azure'],
    'ml engineer'     : ['ml ', 'machine learning', 'ai ', 'artificial intelligence', 'nlp', 'vision'],
}

def rule_based_filter(df: pd.DataFrame) -> pd.DataFrame:
    """Keep rows where title contains at least one keyword for the search_role."""
    def is_valid(row):
        title = str(row['title']).lower()
        role  = str(row['search_role']).lower()
        for key, kws in KEYWORD_MAP_P7.items():
            if key in role:
                return any(kw in title for kw in kws)
        role_words = role.replace('_', ' ').split()
        return any(word in title for word in role_words if len(word) > 3)
    return df[df.apply(is_valid, axis=1)].reset_index(drop=True)

## Run All 7 Pipelines

In [9]:
results = []

# ── P1: Baseline — no filtering ─────────────────────────────────────────────
print("=" * 65)
print("P1 — Baseline (No Filter)")
print("=" * 65)
results.append(safe_train_evaluate(df_base, 'P1 — Baseline (No Filter)'))

# ── P2–P6: Cosine Similarity with varying N-gram ranges ─────────────────────
ngram_configs = [
    ('P2', (1, 1), 'Cosine Sim TF-IDF N-gram (1,1)'),
    ('P3', (1, 2), 'Cosine Sim TF-IDF N-gram (1,2)'),
    ('P4', (1, 3), 'Cosine Sim TF-IDF N-gram (1,3)'),
    ('P5', (1, 4), 'Cosine Sim TF-IDF N-gram (1,4)'),
    ('P6', (1, 5), 'Cosine Sim TF-IDF N-gram (1,5)'),
]

for pid, ngram, label in ngram_configs:
    print(f"\n{'=' * 65}")
    print(f"{pid} — {label}")
    print(f"{'=' * 65}")
    df_filtered = cosine_filter(df_base, ngram_range=ngram, threshold=COSINE_THRESHOLD)
    results.append(safe_train_evaluate(df_filtered, f'{pid} — {label}'))

# ── P7: Rule-Based Keyword Mapping ──────────────────────────────────────────
print(f"\n{'=' * 65}")
print("P7 — Rule-Based Keyword Mapping")
print(f"{'=' * 65}")
df_p7 = rule_based_filter(df_base)
results.append(safe_train_evaluate(df_p7, 'P7 — Rule-Based Keyword Mapping'))

P1 — Baseline (No Filter)
[P1 — Baseline (No Filter)]  Rows after filter: 4,449  |  Used for training: 4,449  |  Accuracy: 0.4528

P2 — Cosine Sim TF-IDF N-gram (1,1)
[P2 — Cosine Sim TF-IDF N-gram (1,1)]  Rows after filter: 1,574  |  Used for training: 1,574  |  Accuracy: 0.6635

P3 — Cosine Sim TF-IDF N-gram (1,2)
[P3 — Cosine Sim TF-IDF N-gram (1,2)]  Rows after filter: 889  |  Used for training: 889  |  Accuracy: 0.7360

P4 — Cosine Sim TF-IDF N-gram (1,3)
[P4 — Cosine Sim TF-IDF N-gram (1,3)]  Rows after filter: 516  |  Used for training: 516  |  Accuracy: 0.7308

P5 — Cosine Sim TF-IDF N-gram (1,4)
[P5 — Cosine Sim TF-IDF N-gram (1,4)]  Rows after filter: 308  |  Used for training: 308  |  Accuracy: 0.7903

P6 — Cosine Sim TF-IDF N-gram (1,5)
  [P6 — Cosine Sim TF-IDF N-gram (1,5)] Dropping sparse roles (< 2 samples): {'Frontend Developer'}
[P6 — Cosine Sim TF-IDF N-gram (1,5)]  Rows after filter: 197  |  Used for training: 196  |  Accuracy: 0.7250

P7 — Rule-Based Keyword Mappin

## Results Summary

In [10]:
summary_df = pd.DataFrame(results)
summary_df['rows_dropped']  = len(df_base) - summary_df['remaining_rows']
summary_df['drop_rate (%)'] = ((summary_df['rows_dropped'] / len(df_base)) * 100).round(2)

display_cols = ['pipeline', 'remaining_rows', 'rows_dropped', 'drop_rate (%)',
                'rows_used_for_training', 'accuracy']

print("\n" + "=" * 80)
print(" ABLATION STUDY — FINAL SUMMARY")
print("=" * 80)

valid_acc = pd.to_numeric(summary_df['accuracy'], errors='coerce')
display(
    summary_df[display_cols].style
    .set_caption("Empirical Ablation Study: 7 Pipeline Comparison")
    .format({'accuracy': lambda x: f'{x:.4f}' if pd.notna(x) else 'N/A'})
    .highlight_max(subset=['accuracy'], color='#c6efce')
    .highlight_min(
        subset=['accuracy'],
        color='#ffc7ce',
        props='color: black;'
    )
)


 ABLATION STUDY — FINAL SUMMARY


,pipeline,remaining_rows,rows_dropped,drop_rate (%),rows_used_for_training,accuracy
0,P1 — Baseline (No Filter),4449,0,0.000000,4449,0.4528
1,"P2 — Cosine Sim TF-IDF N-gram (1,1)",1574,2875,64.620000,1574,0.6635
2,"P3 — Cosine Sim TF-IDF N-gram (1,2)",889,3560,80.020000,889,0.7360
3,"P4 — Cosine Sim TF-IDF N-gram (1,3)",516,3933,88.400000,516,0.7308
4,"P5 — Cosine Sim TF-IDF N-gram (1,4)",308,4141,93.080000,308,0.7903
5,"P6 — Cosine Sim TF-IDF N-gram (1,5)",197,4252,95.570000,196,0.7250
6,P7 — Rule-Based Keyword Mapping,1742,2707,60.850000,1742,0.7163


In [11]:
numeric_acc = pd.to_numeric(summary_df['accuracy'], errors='coerce')
best_idx  = numeric_acc.idxmax()
best_row  = summary_df.loc[best_idx]
worst_idx = numeric_acc.idxmin()
worst_row = summary_df.loc[worst_idx]

print(f"\n{'='*65}")
print(f"🏆  Best  Pipeline : {best_row['pipeline']}")
print(f"    Remaining Rows  : {best_row['remaining_rows']:,}")
print(f"    Accuracy        : {best_row['accuracy']:.4f}")
print(f"\n💡  Worst Pipeline : {worst_row['pipeline']}")
print(f"    Remaining Rows  : {worst_row['remaining_rows']:,}")
print(f"    Accuracy        : {worst_row['accuracy']:.4f}")
print(f"{'='*65}")


🏆  Best  Pipeline : P5 — Cosine Sim TF-IDF N-gram (1,4)
    Remaining Rows  : 308
    Accuracy        : 0.7903

💡  Worst Pipeline : P1 — Baseline (No Filter)
    Remaining Rows  : 4,449
    Accuracy        : 0.4528
